
# Catchment Health Metrics (CHM) — Quickstart Notebook

This notebook demonstrates how to run the **CHM** package.

**How to use this notebook**
1. **Edit the paths** in the next cell to match your environment.
2. Run each section step‑by‑step. Heavy/online steps (e.g., data downloads) are guarded by a switch so you can skip them during quick tests.
3. Each section begins with a brief explanation of what the function does and what you should expect as outputs.

# Installation

To use the Catchment Health Metrics library, you need a suitable Python environment with the required scientific and geospatial libraries.

Set up a Base Python Environment, include scientific Python stack:`numpy`, `pandas`, `jupyter`,`matplotlib`

For **Windows users**, the simplest option is to install **Anaconda**, which provides all of these in one package.

Additional dependencies used by the Catchment Health Metrics library are listed in **`requirements.txt`**.

Install them with:

```bash
cd <directory-with-library>
pip install -r requirements.txt

Finally, the catchment health metrics library itself should be installed.
If you have cloned the git repository, you can install from your local copy. 

From the command prompt: 

cd <directory-with-library> 
pip install -e . 

When the installation has completed, the following import statements should run without error

# Requirements: 

Provide a path for `CHM_Work_Space`. This is the working directory where all outputs will be saved.

Provide a catchment shapefile (`Catchment_Shapefile_Path`) and a sites shapefile (`Sites_Shapefile_Path`) that defines the monitoring/site locations.

Both shapefiles should be in an appropriate projected coordinate system. If the catchment file has no CRS or you need to override it, you can specify it in the code using the `catchment_crs` argument (e.g. `catchment_crs="EPSG:3111"`).

To append monitoring data to the sites, the layer given by `Sites_Shapefile_Path` must contain a `Site_id` column whose values match the site name used in the monitoring data files.

The modules are designed to be independent, but in practice the output of one module is often used as the input to another. 

For a first run, it is recommended to run the modules step by step as in this notebook. Otherwise, for each specefic module follow the following steps:

-  `Topography` module should be run for all of other modules
-  For `Vegetation` module run Topography >> Vegetation 
-  For `Connectivity` module run Topography >> Vegetation >> Connectivity
-  For `RUSLE` module run Topography >> Vegetation >> Connectivity >> RUSLE
-  For `Road` module run Topography >> Road
-  For `Fire` module run Topography >> Fire
-  For `DEA Landuse` module run Topography >> DEA Landuse
-  For `Landuse_2023` module run Topography >> Landuse_2023
-  For `Hydroclimates` module run Topography >> Hydroclimate
-  For `Monitoring data` module run Topography >> Monitoring data
-  For `Build report` module run All to get a full report, otherwise optional

## 1) Imports

In [ ]:

# --- Import with short aliases (functions-only package) ---
# NOTE: If these imports fail, ensure you've installed the package in this environment:
#   pip install -e .[dev]
# Version check (use the distribution name, not the package name)
# --- Import with short aliases (functions-only package) ---
import importlib.metadata as _im
try:
    __chm_version = _im.version("catchment-health-metrics")
except Exception:
    import chm as _chm
    __chm_version = getattr(_chm, "__version__", "unknown")
print(f"CHM package version: {__chm_version}")


from chm.topography import dem_and_terrain
from chm.vegetation import VegConfig, veg_indices_and_c_factor
from chm import surface_ground_water_connectivity
from chm.rusle import RusleConfig, rusle_and_sdr_rusle
from chm.hydroclimate_historical import HistoricalConfig, hydroclimate_historical
from chm.hydroclimate_projection import AwralProjConfig, hydroclimate_projections
from chm.bushfire import BushfireConfig, historical_bushfire
from chm.roads import RoadsConfig, national_roads
from chm.landuse_2023 import LanduseRiskConfig, landuse_2023
from chm.dea_landuse import DEALanduseConfig, dea_landuse_change
from chm.monitoring_data import MonitoringConfig, monitoring_data
from chm.generate_report import ReportConfig, build_report

<div style="background-color:skyblue; padding:15px; border-radius:10px">


## 2) Paths & configuration

- `CHM_Work_Space` — where outputs will be written (safe to create).  
- `Catchment_Shapefile_Path` — polygon boundary of your catchment or study area.  
- `Sites_Shapefile_Path` — points/polygons of monitoring sites (optional but recommended).

Below we:
- Try to **auto‑discover** shapefiles under `tests/Input data/` for a quick demo.
- Fall back to **editable placeholders** if none are found.


In [ ]:
from chm import default_paths
# --- Workspace (safe to create anywhere) ---
# Edit this to your own output directory
CHM_Work_Space = r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2025\Project\Catchmnet Health Metrics\Output"
Catchment_Shapefile_Path = r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2025\Project\Catchmnet Health Metrics\Input\Coliban River\Catchment boundary\Coliban River.shp"
Sites_Shapefile_Path = r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2025\Project\Catchmnet Health Metrics\Input\Coliban River\Sampling sites\Coliban River.shp"

<div style="background-color:skyblue; padding:15px; border-radius:10px">


## 3) DEM & Terrain

- The DEM and Terrain module automates end-to-end topographic pre-processing for a catchment and its monitoring sites.
- It acquire a DEM (either from a web coverage service or an existing file), clip and reproject it to the catchment, and derive key terrain attributes including slope (degrees, radians, percent), aspect (degrees, radians), Topographic Position Index (TPI), Terrain Ruggedness Index (TRI), and a stream network with Strahler order.
- Beyond catchment products, the module produces per-site summaries and maps. If sites are points, it delineates each point’s upslope contributing catchment and summarises terrain statistics inside it. If sites are polygons, it treats them as sub-catchments and summarises terrain within those boundaries. Outputs are saved with a consistent folder hierarchy for easy hand-over and reproducibility.
- In short, this module provides the terrain layer foundations required by downstream hydrological, erosion, and connectivity analyses, while staying robust for both fully automated and partially user-provided data pipelines.
  
**Inputs**: Catchment boundary, sites (optional), DEM source configured in the function.  
**Outputs**: Slope, aspect, ruggedness, topographic indices, and per‑site summaries (GeoTIFFs/CSV/GPKG, depending on implementation)..


In [ ]:
DEM_url = "https://services.ga.gov.au/gis/services/DEM_SRTM_1Second_Hydro_Enforced_2024/MapServer/WCSServer"
DEM_Input_Path = None  # or r"C:\...\DEM.tif" # if you have a dem layer and wnat to use it, provide a path here

dem_and_terrain(
    CHM_Work_Space=CHM_Work_Space,
    Catchment_Shapefile_Path=Catchment_Shapefile_Path,
    Sites_Shapefile_Path=Sites_Shapefile_Path,
    DEM_Input_Path=DEM_Input_Path,
    DEM_url=DEM_url,
    wcs_tile_cols_rows=(2, 2), # change this for tiling the bbox for large catchmnets
    target_res_m=30.0,
    stream_target_area_ha=60.0,
    catchment_crs=None,  # ("EPSG:3308" for GDA94 / NSW Lambert,  and "EPSG:3111" for  (Vicgrid94))
)

<div style="background-color:skyblue; padding:15px; border-radius:10px">


## 4) Vegetation NDVI & C‑factor

- The Vegetation Indices and C-Factor module automates the retrieval and processing of satellite imagery from Digital Earth Australia (DEA) via STAC, computes NDVI and an empirical C-factor (for RUSLE/erosion modelling), and generates annual products, riparian assessments, and per-site statistics and plots.
- It supports both Landsat 7 (pre-2016) and Sentinel-2 (post-2016) imagery, harmonising band names and resolution to a common DEM grid for spatial consistency across the Catchment Health Metrics (CHM) workflow.- 
The module produces: (i) timestep NDVI/C rasters (for provenance and detailed review), (ii) annual NDVI median and annual C-factor rasters, (iii) catchment-level annual means appended to the catchment GPKG, (iv) riparian NDVI per stream segment and by stream order × year summaries with maps and time-series plots, and (v) per-site clipped rasters, statistics, and figures.

**Inputs**: Catchment, sites, time window and filters (e.g., cloud cover).  
**Outputs**: Index rasters, C‑factor rasters and site‑level summaries.


In [ ]:
from chm.vegetation import VegConfig, veg_indices_and_c_factor

cfg = VegConfig(
    chm_workspace=CHM_Work_Space,
    catchment_path=Catchment_Shapefile_Path,
    sites_path=Sites_Shapefile_Path,
    datetime_range="2010-01-01/2025-10-25",
    catchment_crs=None, #"EPSG:3308"
    cloud_cover_lt=20,
    riparian_buffer_m=30.0,
)
veg_indices_and_c_factor(cfg)


<div style="background-color:skyblue; padding:15px; border-radius:10px">

## 5) Surface & Groundwater Connectivity

This module quantifies how easily sediment and water move across a catchment toward streams, and how vegetation cover modifies that connectivity via the C-factor. It produces spatial layers (rasters) and site-level summaries to support erosion-risk screening, rehabilitation targeting, and temporal reportin.


### Functionality

- Calculate core terrain metrics from a DEM: flow direction/accumulation, Topographic Wetness Index (TWI), and LS (slope length–gradient) factor.  
- Build a stream mask from a contributing-area threshold.  
- Convert annual vegetation C-factor rasters into **SDR (Sediment Delivery Ratio)** per year via a logistic transform of a connectivity index (IC), following a Hamel-style approach.  
- Export catchment-wide rasters (TWI, LS, SDR per year, plus support layers) and per-site clipped rasters and plots.  
- Generate **NDVI–SDR exposure profiles** and **AUC** time series per site (diagnostics of how vegetation pixels distribute across SDR gradients .ata.g#pkg`.

### Capabilities

- Fully automated foldering and reproducible outputs per catchment.  
- Works with both **point sites** (for plotting context) and **polygonal site areas** (for clipping/summarising) when available in the base sites package.  
- Tunable thresholds/parameters for stream definition and the SDR logistic curve.
ic curve.


In [ ]:
surface_ground_water_connectivity(
    CHM_Work_Space,
    Catchment_Shapefile_Path,
    Sites_Shapefile_Path,
    catchment_crs=None, #("EPSG:3308" for GDA94 / NSW Lambert)
    sdr_max=0.8, 
    ic0=0.5,
    k=1.0,
    stream_area_threshold_m2=1.3e4,
)

<div style="background-color:skyblue; padding:15px; border-radius:10px">

## 6) RUSLE & SDR‑RUSLE

The module computes annual soil loss and delivered sediment estimates at **catchment** and **site** scales using:

#### RUSLE (Revised Universal Soil Loss Equation)

RUSLE = R * K * LS * C * P

Where:

- **A** — soil loss *(t/ha/yr)*  
- **R** — rainfall erosivity  
- **K** — soil erodibility  
- **LS** — slope length–gradient factor  
- **C** — cover management  
- **P** — support practice  

Download R, C, P, K from https://data.csiro.au/collection/csiro:19354v1 

#### SDR-RUSLE (Delivered Sediment)
SDR-RUSLE = RUSLE * SDR

This uses the annual **SDR rasters** generated by the Connectivity module.

### Processing Summary
The module:
- Harmonises external factor rasters (**K**, **P**, **R**) to the DEM grid.  
- Iterates over annual **C-factor** and **SDR** rasters.  
- Writes **CSV tables**, **plots**, and **GeoPackage layers** suitable for reporting and dashboards.

In [ ]:
cfg = RusleConfig(
    chm_workspace=CHM_Work_Space,
    catchment_path=Catchment_Shapefile_Path,
    sites_path=Sites_Shapefile_Path,
    k_factor_path=r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2025\Project\Catchmnet Health Metrics\Input\C K P R factors\k_factor_g94.tif",
    p_factor_path=r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2025\Project\Catchmnet Health Metrics\Input\C K P R factors\p_factor_g94.tif",
    r_factor_path=r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2025\Project\Catchmnet Health Metrics\Input\C K P R factors\r_factor_g94.tif",
    catchment_crs=None,
)

rusle_and_sdr_rusle(cfg)

<div style="background-color:skyblue; padding:15px; border-radius:10px">

## 7) National Roads Exposure Profile

This module quantifies how **national roads** intersect with **hydrological connectivity** and **wetness** across sites.  
It produces **cumulative exposure profiles** (Road area vs. SDR/TWI), computes **normalised AUC metrics** per year.

### Data and Processing

The module:
- Ingests **national road geometries** (from a local file or the ArcGIS REST service).  
- Clips roads first to the **catchment**, then to each **site**.  
- Evaluates their exposure against surface connectivity rasters:

  - **SDR (Sediment Delivery Ratio)** — annual rasters generated by the Connectivity module.  
  - **TWI (Topographic Wetness Index)** — single raster generated by the Connectivity module.  

### Analytical Workflow

For each **site** and each **SDR year**, the tool:
- Builds a **cumulative exposure curve** — cumulative percentage of road-covered pixels within the site (*y-axis*) versus SDR value (*x-axis*).  
- Computes a **normalised AUC (area under the curve)** within a fixed SDR range (global min/max across the catchment), providing a consistent annual time series per site.  
- Generates one **Roads vs. TWI** curve (no AUC time series).

### Outputs

- **Per-site plots** showing exposure profiles and AUC trends.  
- **Per-site CSVs** (produced via supporting modules).

### Interpretation
- The **AUC** condenses the full curve into a single comparable number between **0–1** (after normalisation).  
- **Higher AUC** → road pixels (area) are concentrated at **higher SDR/TWI values**, meaning a larger share of site area is road and in **more connected or wetter locations**, given the fixed domain.

In [ ]:
cfg = RoadsConfig(
    chm_workspace=CHM_Work_Space,
    catchment_path=Catchment_Shapefile_Path,
    road_input_path=None,  # or a local GPKG/GeoJSON/SHP
    make_plots=True,
)
national_roads(cfg)


<div style="background-color:skyblue; padding:15px; border-radius:10px">

## 8) Historical Bushfire Exposure Profile

This module quantifies **bushfire exposure** within each monitoring site by relating burned areas to:

- **(a)** annual **Sediment Delivery Ratio (SDR)** rasters, and  
- **(b)** a static **Topographic Wetness Index (TWI)** raster.  

Results are summarised as **cumulative exposure profiles** and **normalised Area-Under-Curve (AUC)** metrics written back to the project’s `Sites Data.gpkg`.

### Workflow Overview

This module builds a reproducible, geospatial workflow to:

1. **Acquire** historical bushfire polygons — from a local file or ArcGIS REST service.  
2. **Clip** them to the target **catchment** and then to **per-site polygons**.  
3. For each SDR raster year *Y*, **select fires** whose ignition dates fall within a rolling window  
   \[
   [Y - (W - 1),\; Y]
   \]
   where *W = window_years* (default = 5 years).  
4. **Rasterise** the windowed fire polygons to match the SDR/TWI grid at each site.  
5. **Construct cumulative exposure curves** — percentage of site area burned (*y-axis*) versus SDR or TWI value (*x-axis*).  
6. **Compute normalised AUC** over a fixed, catchment-wide x-axis range to ensure comparability across years and sites.  

### Interpretation
- The **AUC** condenses the full curve into a single comparable number between **0–1** (after normalisation).  
- **Higher AUC** → burned pixels are concentrated at **higher SDR/TWI values**, meaning a larger share of site area burned in **more connected or wetter locations**, given the fixed domain.

In [ ]:
cfg = BushfireConfig(
    chm_workspace=CHM_Work_Space,
    catchment_path=Catchment_Shapefile_Path,
    bushfire_input_path=None,      # or a local GPKG/GeoJSON/Shapefile if you have it
    window_years=5,
    make_plots=True,
)

historical_bushfire(cfg)


<div style="background-color:skyblue; padding:15px; border-radius:10px">

## 9) DEA Lnaduse change
This module automates **land-cover trend analysis** using **Geoscience Australia’s DEA Land Cover annual mosaics**.  
It provides a reproducible, transparent, and reporting-ready workflow for tracking changes in land cover across catchments and sites.

### Key Features

- **Reproducibility** — deterministic file naming and a consistent folder hierarchy ensure traceable and repeatable outputs.  
- **Comparability** — stable class codes and percent composition enable consistent analysis across years and projects.  
- **Communication-ready outputs** — generates maps, time series, and CSV summaries suitable for immediate inclusion in technical reports.

### Typical Use Cases

- **Baseline condition summaries** for environmental reporting.  
- **Land-use change screening** to detect trends over time.  

In [ ]:
cfg = DEALanduseConfig(
    chm_workspace=CHM_Work_Space,
    catchment_path=Catchment_Shapefile_Path,
    start_year=2010,
    end_year=2025,
    download_if_missing=True,
    dea_level="level3",  # change to "level4" if/when available on server
)
dea_ds, sites_ds = dea_landuse_change(cfg)

<div style="background-color:skyblue; padding:15px; border-radius:10px">

## 10) Landuse 2023 Exposure Profiles
This module quantifies how **different land-use types** — either fine classes or **10 consolidated groups** — intersect with **sediment delivery risk (SDR)** across the catchment and within monitoring sites.

It produces **exposure curves** (cumulative area vs. SDR), **normalised AUC metrics** per year, and **time-series plots** that capture how each land-use class contributes to sediment delivery over time.

### Key Characteristics

- Evaluates **land-use–SDR interactions** for land-use classifications.  
- Computes **annual exposure profiles** and **AUC metrics** per site, enabling year-to-year comparisons.  
- Generates **time-series plots** showing how the spatial exposure of each land-use class evolves.  

### Outputs

- **Per-site exposure plots** and **AUC time series** for each land-use class.    
- Ready-to-publish figures for technical reporting and dashboards.

### Interpretation
- The **AUC** condenses the full curve into a single comparable number between **0–1** (after normalisation).  
- **Higher AUC** → landuse classes pixels are concentrated at **higher SDR/TWI values**, meaning a larger share of site area is that landcover calss in **more connected or wetter locations**, given the fixed domain.

In [ ]:
# ===================== Example runner (edit paths) =====================
if __name__ == "__main__":
    Landuse_url = (
        "https://di-daa.img.arcgis.com/arcgis/rest/services/"
        "Land_and_vegetation/Catchment_Scale_Land_Use_Agricultural_Industries/ImageServer/exportImage"
    )

    cfg = LanduseRiskConfig(
        chm_workspace=CHM_Work_Space,
        catchment_path=Catchment_Shapefile_Path,
        landuse_vector_path=None,  # set to a vector with 'lu_class' to use Branch A
        image_service_url=Landuse_url,
        image_pixel_size=50,
        request_timeout=180,
        write_site_plots=True,
        write_catchment_plot=True,
    )
    land_folder, sites_folder = landuse_2023(cfg)

<div style="background-color:skyblue; padding:15px; border-radius:10px">

## 11) Historical Hydroclimate

The **Unified Historical Hydroclimate** module ingests and processes long-term **daily hydroclimate datasets** for a target catchment and its monitoring sites.  
It automates data acquisition, spatial clipping, temporal aggregation, and figure generation for reproducible hydroclimate analysis.

### Core Functionality
- **Download** and preprocess long-term daily datasets.  
- **Clip** data to the catchment and each monitoring site geometry.  
- **Aggregate** results to produce catchment-wide and per-site **daily** and **annual** time series.  
- **Generate publication-ready figures** for rainfall, temperature, evapotranspiration, runoff, and soil moisture.

### Supported Datasets

- **AWAP (AGCD)** daily variables:  
  - *Precipitation*  
  - *Minimum temperature*  
  - *Maximum temperature*  

- **AWRAL v7** daily variables:  
  - *Runoff (qtot)*  
  - *Actual evapotranspiration (etot)*  
  - *Upper soil moisture (s0)*  
  - *Deeper soil moisture (sd)*  

### Outputs

- Standardised folder hierarchy under each catchment for consistent storage.  
- Consistent file naming for reproducibility across projects.  
- Time-series CSVs and publication-quality plots for immediate reporting.

In [ ]:
cfg = HistoricalConfig(
    chm_workspace=CHM_Work_Space,
    catchment_path=Catchment_Shapefile_Path,
    start_year=2010,
    end_year=2025,
    make_site_csvs=True,      # per-site outputs
    enable_awap=True,         # include AWAP
    enable_awral=True         # include AWRAL
)

hydroclimate_historical(cfg)

<div style="background-color:skyblue; padding:15px; border-radius:10px">

## 12) Projection Hydroclimates
This module automates **end-to-end retrieval and summarisation** of **AWRAL hydrologic projections** using **NCI THREDDS OPeNDAP** services.  
It streamlines access, spatial subsetting, aggregation, and export of projection data for multiple climate scenarios and hydrological variables.

### Supported Scenarios and Variables

**Projections:**
- *Historical*
- *RCP4.5*
- *RCP8.5*

**Variables:**
- *Runoff*  
- *Actual evapotranspiration (ET)*  
- *Upper soil moisture (s0)*  
- *Deeper soil moisture (sd)*  

### Processing Workflow

1. **Builds OPeNDAP URLs** dynamically from parameterised patterns.  
2. **Subsets** each dataset to the **catchment bounding box** (*EPSG:4326*) and projection time window.  
3. **Writes clipped NetCDFs** per *(projection × variable)* into the project workspace.  
4. **Computes daily catchment means** using a polygon mask (with robust fallback to nearest-cell extraction).  
5. **Optionally computes daily site means** for each monitoring-site polygon.  
6. **Exports tidy CSVs** (catchment- and site-level) with clear column names:  

In [ ]:
cfg = AwralProjConfig(
    chm_workspace=CHM_Work_Space,
    catchment_path=Catchment_Shapefile_Path,
    make_site_csvs=True,  # set False if you only want the catchment CSV
)

hydroclimate_projections(cfg)

<div style="background-color:skyblue; padding:15px; border-radius:10px">

## 13) Appending monitorin data

This module **operationalises field monitoring data** for catchment-scale reporting.  
It joins each site’s Excel file (e.g., `Site_7.xlsx`) to its corresponding **site geometry**, producing per-site GeoPackages and CSVs, along with **time-series plots** for selected variables.

When a variable list is not supplied, the module **auto-detects analytes** directly from each spreadsheet.  
Thresholds can be defined as:
- **Fixed values** (e.g., pH = 7.0), or  
- **Data-driven** (default rule: *WQT = 1.5 × mean*).

### Key Capabilities

- **Robust per-site ingestion** of Excel-based monitoring data with flexible column naming conventions.  
- **Auto-detection** of analyte columns — prioritises common water-quality variables, then includes other numeric fields.  
- **Site-specific statistics** including:
  - Mean, Median, Minimum, Maximum  
  - Standard deviation, Variance, Mode  
  - Data gaps, WQT threshold, and proportion of exceedance  
- **Clean plotting style**:
  - Stacked time-series plots with shared x-axis  
  - Labelled mean and WQT reference line  
  - Exceedance percentages annotated per site  
- **Organised outputs**:
  - Results stored under the standard **CHM hierarchical workspace** for full reproducibility.

### Outputs

- Per-site **CSV summaries**  
- **Publication-ready plots** showing time-series and exceedance behaviour  
- Consistent folder naming and reproducible file structure across projects

In [ ]:
cfg = MonitoringConfig(
    chm_workspace=CHM_Work_Space,
    catchment_path=Catchment_Shapefile_Path,
    monitoring_folder=r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2025\Project\Catchmnet Health Metrics\Input\Greater Sydney\Monitoring data",
    variables=["Total Phosphorus", "Total Nitrogen", "Turbidity", "pH"],  # add/remove freely
    thresholds={
        "pH": 7.0,
        # "Turbidity": 25.0,   # optionally fix a specific WQT
        # all others use 1.5 × mean
    },
)
monitoring_data(cfg)

<div style="background-color:skyblue; padding:15px; border-radius:10px">

## 14) Build a report

The **Report Builder (`report_builder.py`)** module assembles a fully formatted **Word report (DOCX)** for a given catchment.  
It automatically discovers and integrates figures, tables, and datasets produced by upstream CHM modules — such as **Hydroclimate Historical**, **Land-use Risk**, **Riparian NDVI**, **RUSLE/SDR-RUSLE**, and **Monitoring Append** — into a single, coherent technical report.

The module also performs limited post-processing, computing a small set of **derived metrics** (e.g., annual aggregates, SPI/SRI indices) before embedding results into a narrative report structure.

### What This Module Does

- **Loads** unified daily hydroclimate CSVs (or legacy AWAP/AWRAL data if available).  
- **Computes** annual aggregates, including derived **Mean Temperature**.  
- **Optionally calculates** drought indices **SPI-12** and **SRI-12** (requires `standard_precip`).  
- **Composes** two-panel comparison figures for:
  - **DEA Land Cover** (first vs. last year)
  - **Riparian NDVI** (first vs. last year)
- **Auto-discovers and inserts** available figures, tables, and CSVs from catchment and site folders.  
- **Generates per-site report sections** that include:
  - Monitoring plots and summaries  
  - Land-use and erosion time-series  
  - Site-level hydroclimate, SPI, and SRI trends  
- **Writes** a complete, publication-ready **DOCX report** into the catchment workspace.

### Outputs

- A polished **Word report (`.docx`)** containing:
  - Catchment-scale overview, datasets, and maps  
  - Time-series plots and summaries for all sites  
  - Integrated land, vegetation, and erosion metrics  
- **Fully automated layout** — minimal manual editing required.  
- Ready for direct submission to clients or publication in project deliverables.

In [ ]:
# ===================== quick runner =====================
if __name__ == "__main__":
    cfg = ReportConfig(
        chm_workspace=CHM_Work_Space,
        catchment_path=Catchment_Shapefile_Path,
        enable_spi_sri=True,   # set False if standard_precip is not installed
    )
    build_report(cfg)